# Notebook 07 - Phase 2: Classification et Reranking

Objectif de ce notebook:
- entrainer un classifieur de categorie (documents + queries),
- integrer la prediction de categorie dans le pipeline de retrieval,
- comparer l'impact sur TF-IDF, BM25+ et Embeddings,
- generer la soumission Phase 2 au format Kaggle.


## 1) Imports et chargement des donnees


In [1]:
import json
import os
import pickle
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

DATA_DIR = "../data"
MODELS_DIR = "../models"
OUTPUTS_DIR = "../outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

with open(os.path.join(DATA_DIR, "queries_train.json"), "r", encoding="utf-8") as f:
    queries_train = json.load(f)
with open(os.path.join(DATA_DIR, "queries_test.json"), "r", encoding="utf-8") as f:
    queries_test = json.load(f)
with open(os.path.join(DATA_DIR, "qgts_train.json"), "r", encoding="utf-8") as f:
    qgts_train = json.load(f)

df_docs = pd.read_pickle(os.path.join(DATA_DIR, "df_docs_preprocessed.pkl"))

with open(os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl"), "rb") as f:
    tfidf_vectorizer = pickle.load(f)
with open(os.path.join(MODELS_DIR, "tfidf_matrix.pkl"), "rb") as f:
    tfidf_matrix = pickle.load(f)
with open(os.path.join(MODELS_DIR, "bm25_index.pkl"), "rb") as f:
    bm25_index = pickle.load(f)

corpus_embeddings = np.load(os.path.join(MODELS_DIR, "corpus_embeddings.npy"))
corpus_embeddings = normalize(corpus_embeddings)
st_model = SentenceTransformer("all-MiniLM-L6-v2")

id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_docs["id"]) }
idx_to_id = {idx: doc_id for doc_id, idx in id_to_idx.items()}
doc_categories = df_docs["category"].values

print(f"Documents: {len(df_docs):,}")
print(f"Queries train: {len(queries_train)}")
print(f"Queries test: {len(queries_test)}")
print(f"Categories: {sorted(df_docs['category'].unique().tolist())}")
print(f"Embeddings shape: {corpus_embeddings.shape}")


/Users/yvngval93/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/yvngval93/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Documents: 216,041
Queries train: 327
Queries test: 141
Categories: ['android', 'gaming', 'programmers', 'tex', 'unix']
Embeddings shape: (216041, 384)


## 2) Fonctions utilitaires


In [2]:
def build_query_text(q, mode="lex"):
    # mode='emb': texte query only (meilleur pour embeddings)
    # mode='lex': texte + tags + title (utile pour lexical + classifieur)
    text = (q.get("text", "") or "").strip()
    title = (q.get("title", "") or "").strip()
    tags = " ".join(q.get("tags", []) or []).strip()

    if mode == "emb":
        return text

    parts = [text, tags, title]
    return " ".join([p for p in parts if p]).strip()


def get_relevant_doc_indices(query_id):
    rel = qgts_train[query_id]["relevant_doc_ids"]
    return [id_to_idx[e["doc_id"]] for e in rel if e["doc_id"] in id_to_idx]


def topk_sorted_indices(scores, k=100):
    if k >= scores.shape[0]:
        return np.argsort(scores)[::-1]
    idx = np.argpartition(-scores, k - 1)[:k]
    return idx[np.argsort(scores[idx])[::-1]]


def evaluate_rankings(rankings, queries, k_eval=10, k_submit=100):
    mrr_list, rec10_list, prec10_list, rec100_list = [], [], [], []

    for i, q in enumerate(queries):
        rel_set = set(get_relevant_doc_indices(q["id"]))
        pred = rankings[i]

        rr = 0.0
        for r, idx in enumerate(pred[:k_eval], start=1):
            if idx in rel_set:
                rr = 1.0 / r
                break

        top10 = set(pred[:k_eval])
        top100 = set(pred[:k_submit])

        mrr_list.append(rr)
        rec10_list.append(len(top10 & rel_set) / max(len(rel_set), 1))
        prec10_list.append(len(top10 & rel_set) / k_eval)
        rec100_list.append(len(top100 & rel_set) / max(len(rel_set), 1))

    return {
        "MRR@10": float(np.mean(mrr_list)),
        "Recall@10": float(np.mean(rec10_list)),
        "Precision@10": float(np.mean(prec10_list)),
        "Recall@100": float(np.mean(rec100_list)),
        "n_queries": len(queries),
    }


## 3) Classification de categorie

Modele choisi: Logistic Regression sur TF-IDF character n-grams (`char_wb`).

Pourquoi ce choix:
- robuste aux fautes et variantes lexicales,
- tres efficace pour distinguer les domaines (android, tex, unix, ...),
- permet des probabilites par classe pour le reranking (soft boost).


In [3]:
q_train_split, q_val_split = train_test_split(
    queries_train,
    test_size=0.25,
    random_state=42,
    stratify=[q["category"] for q in queries_train],
)

sampled_docs = []
for cat, group in df_docs.groupby("category"):
    n = min(len(group), 12000)
    sampled_docs.append(group.sample(n=n, random_state=42))
df_docs_sampled = pd.concat(sampled_docs, ignore_index=True)

X_docs = df_docs_sampled["content_clean"].astype(str).tolist()
y_docs = df_docs_sampled["category"].astype(str).tolist()

X_q_train = [build_query_text(q, mode="lex") for q in q_train_split]
y_q_train = [q["category"] for q in q_train_split]

X_q_val = [build_query_text(q, mode="lex") for q in q_val_split]
y_q_val = [q["category"] for q in q_val_split]

classifier = make_pipeline(
    TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=350000,
        sublinear_tf=True,
    ),
    LogisticRegression(
        max_iter=1000,
        C=4.0,
        solver="saga",
        n_jobs=-1,
    ),
)

classifier.fit(X_docs + X_q_train, y_docs + y_q_train)
val_pred = classifier.predict(X_q_val)
val_acc = accuracy_score(y_q_val, val_pred)

print(f"Validation accuracy (query categories): {val_acc:.4f}")
print(classification_report(y_q_val, val_pred, digits=4))


Validation accuracy (query categories): 0.9878
              precision    recall  f1-score   support

     android     1.0000    0.9286    0.9630        14
      gaming     1.0000    1.0000    1.0000        10
 programmers     1.0000    1.0000    1.0000        13
         tex     1.0000    1.0000    1.0000        33
        unix     0.9231    1.0000    0.9600        12

    accuracy                         0.9878        82
   macro avg     0.9846    0.9857    0.9846        82
weighted avg     0.9887    0.9878    0.9878        82



## 4) Retrieval + reranking par categorie (TF-IDF, BM25+, Embeddings)

On compare 3 variantes par modele:
- baseline retrieval,
- soft reranking (`score + alpha * P(categorie_query = categorie_doc)`),
- hard filtering (on garde uniquement la categorie predite).


In [4]:
def scores_tfidf(query_text_lex):
    q_vec = tfidf_vectorizer.transform([query_text_lex])
    return cosine_similarity(q_vec, tfidf_matrix).ravel()


def scores_bm25(query_text_lex):
    return bm25_index.get_scores(query_text_lex.lower().split())


def scores_embeddings(query_text_emb):
    q_emb = st_model.encode([query_text_emb], convert_to_numpy=True, normalize_embeddings=True)
    return (q_emb @ corpus_embeddings.T).ravel()


def evaluate_model_with_reranking(model_name, queries_eval, alpha=1.0):
    X_eval_lex = [build_query_text(q, mode="lex") for q in queries_eval]

    pred_cat = classifier.predict(X_eval_lex)
    pred_proba = classifier.predict_proba(X_eval_lex)
    class_to_col = {c: i for i, c in enumerate(classifier.classes_)}

    rankings_base = []
    rankings_soft = []
    rankings_hard = []

    for i, q in enumerate(queries_eval):
        q_lex = build_query_text(q, mode="lex")
        q_emb = build_query_text(q, mode="emb")

        if model_name == "TF-IDF":
            base_scores = scores_tfidf(q_lex)
        elif model_name == "BM25+":
            base_scores = scores_bm25(q_lex)
        elif model_name == "Embeddings":
            base_scores = scores_embeddings(q_emb)
        else:
            raise ValueError(f"Unknown model: {model_name}")

        rank_base = topk_sorted_indices(base_scores, k=100)
        rankings_base.append(rank_base)

        bonus = np.zeros_like(base_scores, dtype=np.float32)
        for cat_name, col_idx in class_to_col.items():
            bonus[doc_categories == cat_name] = pred_proba[i, col_idx]

        soft_scores = base_scores + alpha * bonus
        rank_soft = topk_sorted_indices(soft_scores, k=100)
        rankings_soft.append(rank_soft)

        mask = (doc_categories == pred_cat[i])
        hard_scores = np.where(mask, base_scores, -1e9)
        rank_hard = topk_sorted_indices(hard_scores, k=100)
        rankings_hard.append(rank_hard)

    metrics_base = evaluate_rankings(rankings_base, queries_eval)
    metrics_soft = evaluate_rankings(rankings_soft, queries_eval)
    metrics_hard = evaluate_rankings(rankings_hard, queries_eval)

    return {
        "baseline": metrics_base,
        "soft": metrics_soft,
        "hard": metrics_hard,
    }


alpha = 1.0
results = {}
for model_name in ["TF-IDF", "BM25+", "Embeddings"]:
    print(f"Evaluating {model_name}...")
    results[model_name] = evaluate_model_with_reranking(model_name, q_val_split, alpha=alpha)

rows = []
for model_name, d in results.items():
    for variant, m in d.items():
        rows.append({
            "model": model_name,
            "variant": variant,
            "MRR@10": m["MRR@10"],
            "Recall@10": m["Recall@10"],
            "Precision@10": m["Precision@10"],
            "Recall@100": m["Recall@100"],
            "query_category_acc": val_acc,
        })

results_df = pd.DataFrame(rows).sort_values(["MRR@10", "Recall@100"], ascending=False)
results_df


Evaluating TF-IDF...
Evaluating BM25+...
Evaluating Embeddings...


,model,variant,MRR@10,Recall@10,Precision@10,Recall@100,query_category_acc
7,Embeddings,soft,0.501703,0.345537,0.218293,0.643122,0.987805
8,Embeddings,hard,0.501703,0.345537,0.218293,0.643122,0.987805
6,Embeddings,baseline,0.501094,0.346641,0.217073,0.638369,0.987805
5,BM25+,hard,0.184074,0.092696,0.065854,0.245093,0.987805
4,BM25+,soft,0.175557,0.091883,0.063415,0.230459,0.987805
3,BM25+,baseline,0.172575,0.088335,0.059756,0.225055,0.987805
1,TF-IDF,soft,0.140684,0.090209,0.059756,0.242980,0.987805
2,TF-IDF,hard,0.140684,0.090209,0.059756,0.242980,0.987805
0,TF-IDF,baseline,0.127270,0.083768,0.050000,0.228541,0.987805


## 5) Entrainement final (toutes les queries train) + generation soumission

Config finale retenue pour la soumission:
- retrieval: Embeddings (`all-MiniLM-L6-v2`) sur query `text`.
- reranking: soft category boost avec `alpha` tune sur train.
- colonne `category`: prediction du classifieur.


In [5]:
X_q_all = [build_query_text(q, mode="lex") for q in queries_train]
y_q_all = [q["category"] for q in queries_train]

classifier.fit(X_docs + X_q_all, y_docs + y_q_all)
class_to_col = {c: i for i, c in enumerate(classifier.classes_)}

train_proba = classifier.predict_proba(X_q_all)

q_emb_train = st_model.encode(
    [build_query_text(q, mode="emb") for q in queries_train],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

S_train = q_emb_train @ corpus_embeddings.T


def evaluate_alpha(alpha):
    rankings = []
    for i, q in enumerate(queries_train):
        bonus = np.zeros(S_train.shape[1], dtype=np.float32)
        for cat_name, col_idx in class_to_col.items():
            bonus[doc_categories == cat_name] = train_proba[i, col_idx]

        scores = S_train[i] + alpha * bonus
        rankings.append(topk_sorted_indices(scores, k=100))

    m = evaluate_rankings(rankings, queries_train)
    return alpha, m["MRR@10"], m["Recall@100"]


alpha_grid = [0.0, 0.2, 0.5, 1.0, 1.5]
alpha_results = [evaluate_alpha(a) for a in alpha_grid]
alpha_df = pd.DataFrame(alpha_results, columns=["alpha", "MRR@10", "Recall@100"])
alpha_df


,alpha,MRR@10,Recall@100
0,0.0,0.463496,0.581370
1,0.2,0.474158,0.604070
2,0.5,0.476197,0.605089
3,1.0,0.476197,0.605599
4,1.5,0.476197,0.605599


In [6]:
best_alpha = alpha_df.sort_values(["MRR@10", "Recall@100"], ascending=False).iloc[0]["alpha"]
print(f"Selected alpha: {best_alpha}")

X_test_lex = [build_query_text(q, mode="lex") for q in queries_test]
pred_cat_test = classifier.predict(X_test_lex)
pred_proba_test = classifier.predict_proba(X_test_lex)

q_emb_test = st_model.encode(
    [build_query_text(q, mode="emb") for q in queries_test],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

S_test = q_emb_test @ corpus_embeddings.T

rows = []
K_SUBMIT = 100
for i, q in enumerate(queries_test):
    bonus = np.zeros(S_test.shape[1], dtype=np.float32)
    for cat_name, col_idx in class_to_col.items():
        bonus[doc_categories == cat_name] = pred_proba_test[i, col_idx]

    final_scores = S_test[i] + float(best_alpha) * bonus
    top_idx = topk_sorted_indices(final_scores, k=K_SUBMIT)
    top_ids = [idx_to_id[idx] for idx in top_idx]

    rows.append({
        "query_id": q["id"],
        "relevant_doc_ids": json.dumps(top_ids),
        "category": pred_cat_test[i],
    })

submission_df = pd.DataFrame(rows)
submission_path = os.path.join(OUTPUTS_DIR, "phase2_submission_v1.csv")
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved: {submission_path}")
print(f"Shape: {submission_df.shape}")
assert submission_df.shape[0] == len(queries_test)
assert (submission_df["relevant_doc_ids"].apply(lambda x: len(json.loads(x))) == K_SUBMIT).all()
print("Format checks OK")
submission_df.head(5)


Selected alpha: 1.0
Submission saved: ../outputs/phase2_submission_v1.csv
Shape: (141, 3)
Format checks OK


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""c583f3cd-b1ca-4b74-ac3e-1c6771eb6a8e_131332""...",programmers
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""edd58474-cef1-4d75-b184-99ee27def6a2_116280""...",unix
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",android
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""ef46acba-fed8-4bc1-ba2e-df09048d8743_27744"",...",unix
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""59108dd2-5f5a-4add-b947-3303c8aaa891_123331""...",tex
